In [ ]:
%pip install langchain langchain_community langchain_ollama duckduckgo-search

Defaulting to user installation because normal site-packages is not writeable
  Using cached duckduckgo_search-8.1.1-py3-none-any.whl.metadata (16 kB)
Using cached duckduckgo_search-8.1.1-py3-none-any.whl (18 kB)
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/4.0 MB 2.0 MB/s eta 0:00:02
   ---------- ----------------------------- 1.0/4.0 MB 2.3 MB/s eta 0:00:02
   ------------------ --------------------- 1.8/4.0 MB 2.7 MB/s eta 0:00:01
   ---------------------------- ----------- 2.9/4.0 MB 3.2 MB/s eta 0:00:01
   ---------------------------------------  3.9/4.0 MB 3.6 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 3.5 MB/s  0:00:01
   ---------------------------------------- 0.0/5.2 MB ? eta -:--:--
   ---------- ----------------------------- 1.3/5.2 MB 6.6 MB/s eta 0:00:01
   -------------------- ------------------- 2.6/5.2

In [5]:
from langchain_community.llms import Ollama
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

In [7]:
# We specify the model we pulled in Step 1
llm = Ollama(model="llama3:8b")


In [9]:
import sys
!{sys.executable} -m pip install -U ddgs

Defaulting to user installation because normal site-packages is not writeable
  Using cached ddgs-9.16.0-py3-none-any.whl.metadata (16 kB)
Using cached ddgs-9.16.0-py3-none-any.whl (47 kB)


In [10]:
search = DuckDuckGoSearchRun()

In [11]:
# This is the prompt template, our instruction manual for the LLM
prompt = ChatPromptTemplate.from_template(
    """You are a helpful AI assistant. You must answer the user's question 
    based *only* on the following search results. If the search results 
    are empty or do not contain the answer, say 'I could not find 
    any information on that.'

    Search Results:
    {context}

    Question:
    {question}
    """
)

In [12]:
# This is our RAG chain
chain = (
    RunnablePassthrough.assign(
        # "context" is a new key we add to the dictionary.
        # Its value is the *output* of running the 'search' tool
        # with the original 'question' as input.
        context=lambda x: search.run(x["question"])
    )
    | prompt  # The dictionary (now with 'context' and 'question') is "piped" into the prompt
    | llm     # The formatted prompt is "piped" into the LLM
)

In [ ]:
print("🤖 Hello! I'm a real-time AI assistant. What's new?")
while True:
    try:
        user_query = input("You: ")
        if user_query.lower() in ["exit", "quit"]:
            print("🤖 Goodbye!")
            break
        
        print("🤖 Thinking...")
        
        # This one line runs the whole RAG process
        response = chain.invoke({"question": user_query})
        
        print(f"🤖: {response}")

    except Exception as e:
        print(f"An error occurred: {e}")

🤖 Hello! I'm a real-time AI assistant. What's new?
🤖 Thinking...
🤖: I could not find any information on that.
